# Appendix C — How stable are the comparisons?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/C_robustness.ipynb)

Chapters 10 and 11 use a fixed split to teach the methods. Here we investigate the unexpected training-size
effect and repeat the missing/noisy-label experiments. The targets remain ASPCAP labels, not independent truth.
This appendix is more expensive than the worked examples: it fits several dozen models.

**Protocol.** Select continuum pixels using only training stars, before examining validation or test scores.
For the size experiment use three splits and nested training subsets, with a common label scaling within each
split so that coefficients and scatter can be exchanged. For missing and noisy labels use three random draws
on the first split. Retune ridge on validation stars in each experiment. The low-S/N sample is reserved for
reporting, never tuning. Paired bootstrap intervals resample held-out stars together for both methods; they
describe sampling uncertainty conditional on the fitted models. Repeated splits/draws separately show training
variation; three repetitions cannot establish population-wide robustness. The intervals are pointwise, not
corrected for multiple comparisons, and the shared test stars make estimates across splits correlated.

We print bias, robust scatter and RMSE per label and metallicity bin. Bias and RMSE are necessary because a
narrow distribution of differences can still be biased. The initial experiment had no pass/fail requirement
on which method won. The measured regression bounds below were frozen after inspecting that run; they protect
the worked example against changes and are not hypothesis tests.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}
# --- More of The Cannon (Chapter 10): any polynomial order, censoring, L1 regularisation, several starting points,
# --- and a test of whether labels lie inside the training set -------------------------------------------------
from itertools import combinations_with_replacement
import warnings
from scipy.optimize import linprog


def label_terms(K, order=2):
    """The terms of the label vector after the leading 1: tuples of label indices, e.g. (0,), (0, 1), (2, 2, 2)."""
    return [t for p in range(1, order + 1) for t in combinations_with_replacement(range(K), p)]


def vectorize_n(x, order=2):
    """Label vector of polynomial order `order` for scaled labels x (N, K): (N, D). For order 2 it equals vectorize."""
    x = np.atleast_2d(x)
    return np.hstack([np.ones((len(x), 1))] + [np.prod(x[:, list(t)], axis=1, keepdims=True)
                                                for t in label_terms(x.shape[1], order)])


def dvectorize_n(x, order=2):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    T = label_terms(len(x), order)
    J = np.zeros((1 + len(T), len(x)))
    for i, t in enumerate(T):
        for j in range(len(t)):                              # d/dx of x_a x_b x_c ...: drop one factor at a time
            J[1 + i, t[j]] += np.prod(x[list(t[:j] + t[j + 1:])])
    return J


def censor_mask(allowed, order=2):
    """Coefficients allowed at each pixel, (P, D), from the labels allowed at each pixel, (P, K): a term may be
    non-zero only if all the labels in it are allowed. The leading coefficient (the mean spectrum) always is."""
    T = label_terms(allowed.shape[1], order)
    return np.hstack([np.ones((len(allowed), 1), bool)] + [allowed[:, list(t)].all(axis=1, keepdims=True) for t in T])


def fit_theta_l1(V, flux, var, lam=0.0, mask=None, theta0=None, chunk=2048, tol=1e-8, max_iter=20000, kappa=0.1):
    """At every pixel minimise chi^2 + lam * sum_{d >= 1} |theta_d| (Casey et al. 2016), with theta_d = 0 where
    mask (P, D) is False. lam = 0: weighted least squares, solved directly. lam > 0: ADMM (Boyd et al. 2011, §6.4) on rescaled coefficients,
    all pixels of a chunk at once, started from theta0 (P, D) if given. Weights 1/var (N, P); zero-weight entries drop
    out. Returns theta (P, D) and the number of ADMM iterations used (0 for lam = 0)."""
    N, D = V.shape
    P = flux.shape[1]
    mask = np.ones((P, D), bool) if mask is None else mask
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)
    VV = (V[:, :, None] * V[:, None, :]).reshape(N, D * D).T
    pen = np.r_[0.0, np.ones(D - 1)]                             # theta_0 (the mean spectrum) is not penalised
    theta, n_used = np.zeros((P, D)), 0
    for p0 in range(0, P, chunk):
        sl = slice(p0, p0 + chunk)
        A = (VV @ w[:, sl]).T.reshape(-1, D, D)                  # V^T W V at each pixel
        b = (V.T @ (w[:, sl] * flux[:, sl])).T                   # V^T W f
        m = mask[sl]
        if lam == 0:                                             # censored rows/columns replaced by the identity
            Am = A * (m[:, :, None] & m[:, None, :]) + np.eye(D) * ~m[:, None, :]
            theta[sl] = np.einsum('pde,pe->pd', np.linalg.pinv(Am), b * m) * m
            continue
        # Rescale each coefficient so that the curvature of chi^2 along it is 1: theta = d * y. Without this, linear and
        # quadratic terms have curvatures that differ by orders of magnitude and ADMM crawls. |theta_k| = d_k |y_k|.
        dg = np.einsum('pdd->pd', A)
        d = 1 / np.sqrt(np.where(dg > 0, 2 * dg, 1.0))
        As, bs = A * d[:, :, None] * d[:, None, :], b * d
        # ADMM: split y (quadratic part) from z (L1 part and mask), y = z; u is the scaled dual variable. The step
        # size rho scales with the soft threshold lam * d, which in these units sets how far the solution moves.
        rho = np.minimum(kappa * lam * np.median(d, axis=1), 1.0)[:, None]   # capped at the (unit) curvature
        M = np.linalg.inv(2 * As + rho[:, :, None] * np.eye(D))   # factorised once, used at every iteration
        z = np.zeros_like(b) if theta0 is None else theta0[sl] * m / d
        u = np.zeros_like(b)
        thr = pen * lam * d / rho                                # soft threshold for each scaled coefficient
        act = np.arange(len(b))                                  # pixels not yet converged
        for it in range(max_iter):
            y = np.einsum('pde,pe->pd', M[act], 2 * bs[act] + rho[act] * (z[act] - u[act]))
            v = y + u[act]
            z_new = np.where(m[act], np.sign(v) * np.maximum(np.abs(v) - thr[act], 0), 0.0)
            da = d[act]                                          # convergence judged on theta, not on y
            done = (np.abs(da * (y - z_new)).max(axis=1) < tol) & (np.abs(da * (z_new - z[act])).max(axis=1) < tol)
            z[act], u[act] = z_new, v - z_new
            act = act[~done]
            if len(act) == 0:
                break
        if len(act):
            warnings.warn(f'fit_theta_l1: {len(act)} pixels did not converge in {max_iter} iterations')
        n_used = max(n_used, it + 1)
        theta[sl] = z * d                                        # z has the exact zeros
    return theta, n_used


def median_scatter(resid, err2, n_bisect=50):
    """AnniesLasso's scatter: at each pixel the s >= 0 for which the median of resid^2 / (err2 + s^2) over the stars is
    1 (s = 0 where it is already below 1). Bisection on all pixels at once; zero-weight entries are ignored."""
    z = lambda s: np.nanmedian(np.where(np.isfinite(err2), resid**2 / (err2 + s**2), np.nan), axis=0)
    lo, hi = np.zeros(resid.shape[1]), np.full(resid.shape[1], 1.0)
    for _ in range(n_bisect):                                # median chi^2 decreases with s
        mid = (lo + hi) / 2
        big = z(mid) > 1
        lo, hi = np.where(big, mid, lo), np.where(big, hi, mid)
    return np.where(z(0.0) > 1, hi, 0.0)


def train_more(x, flux, err, lam=0.0, mask=None, order=2, n_iter=3, scatter='ml'):
    """Training step with polynomial order, censoring mask and L1 regularisation. scatter='ml': coefficients and
    maximum-likelihood scatter iterated, as in `train` (Chapter 2). scatter='median': AnniesLasso's recipe —
    coefficients fitted once with s = 0, then s from `median_scatter`. Returns theta (P, D) and s (P,)."""
    V, err2 = vectorize_n(x, order), err**2
    if scatter == 'median':
        theta = fit_theta_l1(V, flux, err2, lam, mask)[0]
        return theta, median_scatter(flux - V @ theta.T, err2)
    s, theta = np.zeros(flux.shape[1]), None
    for _ in range(n_iter):
        theta = fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0]     # each fit starts from the previous one
        s = fit_scatter(flux - V @ theta.T, err2)
    return fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0], s


def test_more(flux, err, theta, s, order=2, starts=None):
    """Test step for one spectrum, as `test` (Chapter 2) but for any polynomial order and from several starting
    points (rows of `starts`, scaled labels; default: the median labels, 0); of the fits that converged, the one with
    the lowest chi^2 is kept.
    Returns scaled labels, covariance, chi^2, number of pixels used, converged?"""
    K = next(k for k in range(1, 100) if 1 + len(label_terms(k, order)) == theta.shape[1])
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize_n(x, order)[0]) * w
    jac = lambda x: -(th @ dvectorize_n(x, order)) * w[:, None]
    fits = [least_squares(res, x0, jac=jac, method='lm') for x0 in (np.zeros((1, K)) if starts is None else
                                                                     np.atleast_2d(starts))]
    r = min(fits, key=lambda r: (not r.success, np.sum(r.fun**2)))   # converged fits first, then lowest chi^2
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def in_hull(x, X):
    """Is each row of x (M, K) inside the convex hull of the training labels X (N, K)? True if x is a weighted mean
    of training points with non-negative weights summing to 1: a linear-programming feasibility problem, which works
    in any number of dimensions (a Delaunay triangulation does not, beyond a few). Labels are scaled by the standard
    deviation of X so that kelvin and dex are treated alike."""
    sd = X.std(axis=0)
    A = np.vstack([(X / sd).T, np.ones(len(X))])
    status = np.array([linprog(np.zeros(len(X)), A_eq=A, b_eq=np.r_[p / sd, 1], bounds=(0, None), method='highs').status
                       for p in np.atleast_2d(x)])
    if not np.all((status == 0) | (status == 2)):              # 0: feasible (inside), 2: infeasible (outside)
        raise RuntimeError(f'linprog failed with status {set(status) - {0, 2}}')
    return status == 0
# --- Lux (Horta et al. 2025): labels and spectra both generated from a latent vector per star (Chapter 11) ----
# labels l_n = A z_n + noise (errors sl_n), fluxes f_n = B z_n + noise (errors s_n and pixel scatter s); every step of
# the training is a weighted linear least-squares problem, solved here in closed form.


def lux_weights(values, errors):
    """Inverse variances, 0 where the error is not finite and positive or the value is not finite (masked pixels,
    missing labels), and the values with those entries set to 0 so that they cannot produce NaN."""
    ok = np.isfinite(errors) & (errors > 0) & np.isfinite(values)
    w = np.where(ok, 1 / np.where(ok, errors, 1.0)**2, 0.0)
    return w, np.where(w > 0, values, 0.0)


def lsq_update(G, h, c0, incremental=True, rcond=1e-10):
    """Solve the normal equations G c = h for a stack of problems (G: (K, P, P), h and c0: (K, P)).
    incremental=True: c0 + G^+ (h - G c0), which has the same residual as the minimum-norm solution but keeps the part
    of c0 that the data do not constrain (as a Gauss-Newton step from c0 does); incremental=False: G^+ h.
    Directions with eigenvalues of G below rcond times the largest count as unconstrained (rcond = 0: none do, for G
    that is positive definite, e.g. with a prior)."""
    vals, vecs = np.linalg.eigh(G)
    inv = np.where(vals > rcond * np.max(vals, axis=1, keepdims=True), 1 / np.where(vals > 0, vals, 1.0), 0.0)
    r = h - np.einsum('kpq,kq->kp', G, c0) if incremental else h
    step = np.einsum('kpq,kq,kq->kp', vecs, inv, np.einsum('kqp,kq->kp', vecs, r))
    return c0 + step if incremental else step


def lux_a_step(Z, lab, wl, A, incremental=True):
    """Label coefficients A (M, P) at fixed latents Z (N, P): one weighted least-squares problem per label."""
    G = np.einsum('nm,np,nq->mpq', wl, Z, Z)
    return lsq_update(G, np.einsum('nm,np->mp', wl * lab, Z), A, incremental)


def lux_b_step(Z, flux, w, B, incremental=True):
    """Flux coefficients B (Lambda, P) at fixed latents Z: one weighted least-squares problem per pixel."""
    N, P = Z.shape
    G = ((Z[:, :, None] * Z[:, None, :]).reshape(N, P * P).T @ w).T.reshape(-1, P, P)
    return lsq_update(G, (Z.T @ (w * flux)).T, B, incremental)


def lux_z_step(A, B, lab, wl, flux, w, Z, tau=np.inf, incremental=True):
    """Latents Z (N, P) at fixed A, B: one weighted least-squares problem per star, using its labels (weights wl) and
    fluxes (weights w), with a Gaussian prior of width tau on every latent (ridge term)."""
    P = A.shape[1]
    G = (w @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    G += np.einsum('nm,mp,mq->npq', wl, A, A) + np.eye(P) / tau**2
    rcond = 0.0 if np.isfinite(tau) else 1e-10                # with a prior every direction is constrained by it
    return lsq_update(G, (w * flux) @ B + (wl * lab) @ A, Z, incremental, rcond)


def lux_objective(A, B, Z, s, lab, wl, flux, w, tau=np.inf):
    """Q = -2 ln L (labels and fluxes, up to a constant) + sum |z|^2 / tau^2; masked entries (weight 0) excluded."""
    var = np.where(w > 0, 1 / np.where(w > 0, w, 1.0) + s**2, 1.0)
    q_flux = np.where(w > 0, (flux - Z @ B.T)**2 / var + np.log(var), 0.0)
    return np.sum(wl * (lab - Z @ A.T)**2) + np.sum(q_flux) + np.sum(Z**2) / tau**2


def lux_scatter_step(B, Z, flux, w, s):
    """Pixel scatter by maximum likelihood at fixed B, Z (`fit_scatter` of Chapter 2), kept at each pixel only if it
    does not increase that pixel's part of Q."""
    err2 = np.where(w > 0, 1 / np.where(w > 0, w, 1.0), np.inf)
    resid = flux - Z @ B.T
    q = lambda s_: np.sum(np.where(w > 0, resid**2 / (err2 + s_**2) + np.log(np.where(w > 0, err2, 1.0) + s_**2), 0.0),
                          axis=0)
    s_new = fit_scatter(resid, err2)
    return np.where(q(s_new) <= q(s), s_new, s)


def lux_init(lab, P, n_flux, seed=0):
    """H25 Table 1, eq. 9: z_n = [1, (l_n - median) / ((p97.5 - p2.5) / 4), 0, ...], A and B uniform in [0, 1].
    Pivot and scale come from the labels that are present (NaN = missing); missing entries start at 0."""
    rng = np.random.default_rng(seed)
    q = np.nanpercentile(lab, [2.5, 50, 97.5], axis=0)
    x = np.nan_to_num((lab - q[1]) / ((q[2] - q[0]) / 4))
    Z = np.hstack([np.ones((len(lab), 1)), x, np.zeros((len(lab), P - lab.shape[1] - 1))])
    return rng.uniform(0, 1, (lab.shape[1], P)), rng.uniform(0, 1, (n_flux, P)), Z


def lux_train(lab, lab_err, flux, err, P=49, n_iter=5, tau=1000.0, incremental=True, seed=0):
    """Training as described in H25 §4.2. Stage 1: n_iter rounds of A-, B- and z-steps without scatter or prior.
    Stage 2 (once): scatter, then B with the new scatter, then z with the new B, scatter and the prior.
    Labels missing as NaN or with infinite error are ignored. Returns A, B, s, Z and Q after every step."""
    wl, l0 = lux_weights(lab, lab_err)
    w, f0 = lux_weights(flux, err)
    A, B, Z = lux_init(np.where(wl > 0, lab, np.nan), P, flux.shape[1], seed)
    s, history = np.zeros(flux.shape[1]), []
    for _ in range(n_iter):
        A = lux_a_step(Z, l0, wl, A, incremental)
        B = lux_b_step(Z, f0, w, B, incremental)
        Z = lux_z_step(A, B, l0, wl, f0, w, Z, np.inf, incremental)
        history.append(lux_objective(A, B, Z, s, l0, wl, f0, w))
    s = lux_scatter_step(B, Z, f0, w, s)
    wv = w / (1 + w * s**2)                                   # 1 / (sigma^2 + s^2), 0 where masked
    B = lux_b_step(Z, f0, wv, B, incremental)
    Z = lux_z_step(A, B, l0, wl, f0, wv, Z, tau, incremental)
    history.append(lux_objective(A, B, Z, s, l0, wl, f0, w, tau))
    return A, B, s, Z, np.array(history)


def lux_test(flux, err, A, B, s, rcond=1e-12):
    """Test step (H25 eq. 10): latents of each star from its spectrum alone (no prior), labels = A z.
    Latent directions the spectrum does not constrain (eigenvalues of B^T W B below rcond times the largest) are left at
    zero; the labels do not depend on them only if A has no component along them, which is checked and returned.
    Returns labels (N, M), their formal covariance (N, M, M), and for each star the condition number of the retained
    part of B^T W B and the largest |A v| over the discarded directions v (0 if none)."""
    w, f0 = lux_weights(flux, err)
    wv = w / (1 + w * s**2)
    P = B.shape[1]
    G = (wv @ (B[:, :, None] * B[:, None, :]).reshape(-1, P * P)).reshape(-1, P, P)
    vals, vecs = np.linalg.eigh(G)
    keep = vals > rcond * vals[:, -1:]
    Ginv = np.einsum('npq,nq,nrq->npr', vecs, np.where(keep, 1 / np.where(keep, vals, 1.0), 0.0), vecs)
    z = np.einsum('npq,nq->np', Ginv, (wv * f0) @ B)
    cond = vals[:, -1] / np.min(np.where(keep, vals, np.inf), axis=1)
    leak = np.max(np.where(~keep[:, None, :], np.abs(np.einsum('mp,npq->nmq', A, vecs)), 0.0), axis=(1, 2))
    return z @ A.T, np.einsum('mp,npq,lq->nml', A, Ginv, A), cond, leak
# --- Validation helpers for Appendix C ---------------------------------------------------------------
from numpy.lib.stride_tricks import sliding_window_view


def training_continuum(labels, flux, err):
    """Chapter 3's selection, using ONLY the supplied training stars (three labels, unnormalised spectra)."""
    cq = np.full_like(flux, np.nan)
    for lo, hi in CHIPS:
        c = np.flatnonzero((wave > lo) & (wave < hi))
        width = int(50 / (np.median(wave[c]) * np.log(10) * 6e-6))
        f = np.where(np.isfinite(err[:, c]), flux[:, c], np.nan)
        win = sliding_window_view(f, width, axis=1)[:, ::10]
        quantile = np.nanpercentile(win, 90, axis=2)
        centres = np.arange(win.shape[1]) * 10 + width // 2
        for n in range(len(f)):
            cq[n, c] = np.interp(np.arange(len(c)), centres, quantile[n])
    pivot = np.median(labels, axis=0)
    scale = np.diff(np.percentile(labels, [2.5, 97.5], axis=0), axis=0)[0] / 2
    fq = flux / cq
    eq = np.where(np.isfinite(fq), err / cq, np.inf)
    theta, _, _ = train((labels - pivot) / scale, np.nan_to_num(fq), eq)
    return (np.abs(theta[:, 0] - 1) < 0.015) & np.all(
        np.abs(theta[:, 1:4] / scale) < [1e-5, 0.0045, 0.0085], axis=1)


def robust_std(d):
    """One robust standard deviation per column (label); rows are stars."""
    return 1.4826 * np.median(np.abs(d - np.median(d, axis=0)), axis=0)


def paired_interval(a, b, n_boot=500, seed=42):
    """95% percentile interval for robust_std(a) - robust_std(b), resampling the SAME stars in both.
    Inputs are residuals (stars, labels). The last column is the median scatter over abundances (columns 3:).
    This measures held-out sampling uncertainty conditional on the two fitted models, not training uncertainty.
    """
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        i = rng.integers(len(a), size=len(a))
        sa, sb = robust_std(a[i]), robust_std(b[i])
        draws.append(np.r_[sa - sb, np.median(sa[3:]) - np.median(sb[3:])])
    return np.percentile(draws, [2.5, 97.5], axis=0)
import time
from IPython.display import HTML, display

t0 = time.time()
names = ['teff', 'logg', 'fe_h'] + [e + '_fe' for e in ['c', 'n', 'o', 'mg', 'al', 'si', 'ca', 'mn', 'ni']]
catalogue = load('labels')
data = {}
for sample in ['train', 'test']:
    table = catalogue[catalogue['sample'] == sample].reset_index(drop=True)
    good = np.isfinite(table[names]).all(axis=1).to_numpy()
    f, e = prepare(load('spectra_' + sample))
    data[sample] = (table.loc[good].reset_index(drop=True), f[good], e[good])
tab, raw, rawerr = data['train']
testtab, test_raw, test_rawerr = (a[::2] for a in data['test'])
L, LE = tab[names].to_numpy(), tab[[n + '_err' for n in names]].to_numpy()
truth = testtab[names].to_numpy()
quality = (tab[[n + '_flag' for n in names[3:]]].to_numpy() == 0).all(axis=1)
assert np.all(np.isfinite(LE) & (LE > 0)), 'training label errors must be usable'
print('Stars:', len(L), 'high S/N;', len(truth), 'low S/N; all nine abundance flags clear:', quality.sum())


def fit_cannon(idx, labs, F, E, pivot=None, scale=None):
    if pivot is None:
        pivot = np.median(labs, axis=0)
        scale = np.diff(np.percentile(labs, [2.5, 97.5], axis=0), axis=0)[0] / 2
    th, s = train_more((labs - pivot) / scale, F[idx], E[idx])
    return th, s, pivot, scale


def predict_cannon(model, F, E, starts=None):
    th, s, pivot, scale = model
    out = [test_more(f, e, th, s, starts=starts) for f, e in zip(F, E)]
    assert all(o[4] for o in out), 'test-step convergence'
    return np.array([o[0] for o in out]) * scale + pivot, np.array([o[2] for o in out])


def statistics(pred, ref):
    d = pred - ref
    return pd.DataFrame({'label': names, 'bias': d.mean(axis=0), 'robust': robust_std(d),
                         'rmse': np.sqrt(np.mean(d**2, axis=0))})


def show_table(table, title):
    """Keep the per-label evidence available without making the rendered chapter hundreds of lines long."""
    display(HTML('<details><summary>' + title + '</summary>' +
                 table.to_html(index=False, float_format=lambda x: f'{x:.5g}') + '</details>'))


def compare(a, b, ref):
    sa, sb = robust_std(a - ref), robust_std(b - ref)
    delta = np.r_[sa - sb, np.median(sa[3:]) - np.median(sb[3:])]
    interval = paired_interval(a - ref, b - ref)
    return delta, interval

## C.1 Why can fewer training stars give tighter validation abundances?

Use splits 10 (the original split), 13 and 14. Within each, the 300-star set is a subset of the full set.
Both models use the same continuum selection and label coordinates. On split 10 also exchange the fitted
coefficients and scatter, require all nine abundance flags to be clear, remove stars with mean training
chi-squared above 3, and use three starting points when fitting validation labels. These are diagnostics,
not alternative models selected by their test scores. Quality cuts and residual cuts change the population
as well as its size, so any improvement alone does not identify a unique cause.

In [ ]:
size_results, size_tables = [], []
base = {}
for seed in [10, 13, 14]:
    val = np.random.default_rng(seed).permutation(len(L)) % 5 == 0
    idx = np.flatnonzero(~val)
    small = idx[np.random.default_rng(11).permutation(len(idx))[:300]]
    cont = training_continuum(L[idx, :3], raw[idx], rawerr[idx])
    assert all(cont[(wave > lo) & (wave < hi)].sum() >= 3 for lo, hi in CHIPS)
    F, E = cheb_normalise(raw, rawerr, cont)
    Ft, Et = cheb_normalise(test_raw, test_rawerr, cont)
    pivot = np.median(L[idx], axis=0)
    scale = np.diff(np.percentile(L[idx], [2.5, 97.5], axis=0), axis=0)[0] / 2
    models = {label: fit_cannon(rows, L[rows], F, E, pivot, scale)
              for label, rows in [('full', idx), ('300', small)]}
    if seed == 10:
        full, few = models['full'], models['300']
        models['full theta, 300 scatter'] = (full[0], few[1], pivot, scale)
        models['300 theta, full scatter'] = (few[0], full[1], pivot, scale)
        clean_idx = idx[quality[idx]]
        models['all flags clear'] = fit_cannon(clean_idx, L[clean_idx], F, E, pivot, scale)
        residual = F[idx] - vectorize_n((L[idx] - pivot) / scale) @ full[0].T
        z2 = np.where(np.isfinite(E[idx]), residual**2 / (E[idx]**2 + full[1]**2), np.nan)
        keep = np.nanmean(z2, axis=1) < 3
        models['residual cut'] = fit_cannon(idx[keep], L[idx[keep]], F, E, pivot, scale)
        print('Split 10: continuum pixels', cont.sum(), '; flag-cut stars', len(clean_idx),
              '; residual-cut stars', keep.sum())
    predictions = {}
    for label, model in models.items():
        predictions[label] = [predict_cannon(model, f, e)[0] for f, e in [(F[val], E[val]), (Ft, Et)]]
        print('split', seed, label, 'predictions complete', flush=True)
    if seed == 10:
        starts = [np.zeros(12), np.full(12, -0.5), np.full(12, 0.5)]
        for label in ['full', '300']:
            predictions[label + ', multi-start'] = []
            for f, e in [(F[val], E[val]), (Ft, Et)]:
                single, chi = predict_cannon(models[label], f, e)
                multi, chi_multi = predict_cannon(models[label], f, e, starts)
                assert np.all(chi_multi <= chi + 1e-6), 'multi-start includes the original start'
                predictions[label + ', multi-start'].append(multi)
        base = dict(idx=idx, val=val, F=F, E=E, Ft=Ft, Et=Et, model=models['full'],
                    pred=predictions['full'], cont=cont)
    for label, pred in predictions.items():
        for j, (split, ref) in enumerate([('validation', L[val]), ('test', truth)]):
            delta, ci = compare(pred[j], predictions['300'][j], ref)
            size_results.append(dict(seed=seed, model=label, split=split, delta=delta[-1], lo=ci[0, -1], hi=ci[1, -1]))
            table = statistics(pred[j], ref).assign(seed=seed, model=label, split=split)
            size_tables.append(table)
    print('finished split', seed, 'elapsed seconds', round(time.time() - t0), flush=True)
size_results = pd.DataFrame(size_results)
size_tables = pd.concat(size_tables, ignore_index=True)
print(size_results.round(5).to_string(index=False))
show_table(size_tables, 'Per-label bias, robust scatter and RMSE for the size experiment (K, dex)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, split in zip(axes[0], ['validation', 'test']):
    d = size_results[(size_results.split == split) & (size_results.model != '300')]
    y = np.arange(len(d))
    ax.hlines(y, d.lo, d.hi, color='C0')
    ax.plot(d.delta, y, 'o', color='C0')
    ax.set_yticks(y, [f'{r.model} (split {r.seed})' for r in d.itertuples()])
    ax.axvline(0, color='k', lw=0.8)
    ax.set(xlabel='median abundance scatter minus 300-star model [dex]', title=split)
for ax, split in zip(axes[1], ['validation', 'test']):
    d = size_tables[(size_tables.seed == 10) & (size_tables.split == split)]
    full = d[d.model == 'full'].set_index('label').loc[names[3:]]
    small = d[d.model == '300'].set_index('label').loc[names[3:]]
    for metric in ['robust', 'rmse']:
        ax.plot(range(9), small[metric] / full[metric], 'o-', label=metric)
    ax.axhline(1, color='k', lw=0.8)
    ax.set_xticks(range(9), [n[:-3].capitalize() for n in names[3:]])
    ax.set(xlabel='element', ylabel='300-star / full-model error', title=split + ', split 10')
    ax.legend()
plt.tight_layout()
plt.show()

**Figure C.1.** Top: difference in median robust scatter over nine [X/Fe] labels between each diagnostic model and
the 300-star model, for high-S/N validation stars (left) and the separate S/N 50–100 test stars (right).
Points are measured differences in dex; horizontal bars are paired 95% bootstrap intervals over the stars.
Positive values favour 300 stars. Exchanging scatter at fixed coefficients isolates the effect of test-step
pixel weights; exchanging coefficients includes how training weights affected the learned spectrum. The three
splits test whether the original size comparison persists. Bottom: the 300-star/full-model ratio of robust
scatter and RMSE, element by element on split 10 (both dimensionless). Below one favours the smaller model.
The robust scatter describes the central distribution; RMSE includes bias and large errors.

Figure C.1 shows that on split 10 the full model has 0.0114 dex more median abundance scatter than the 300-star
model. Replacing its
test-time scatter with the 300-star scatter leaves 0.0100 dex; keeping the 300-star coefficients and replacing
their scatter changes the score by only −0.0004 dex. The effect therefore follows the learned coefficients,
rather than mainly the final pixel weights. All-element quality cuts, a residual cut and extra starts leave
the difference. Training weights can still affect the coefficients themselves; these swaps do not separate
that mechanism from differences in sample composition or label correlations.

The other splits give smaller differences (0.0047 and 0.0058 dex), with intervals crossing zero. The low-S/N
differences are also small. Thus the original observation survives several controls but is not evidence for
a generally optimal training size of 300. In fact, on split 10 the smaller model has worse RMSE for five of
the nine abundances, despite tighter robust scatter for all nine: a narrower core does not imply smaller
overall errors. The remaining question is why the learned coefficients change the shape of those residual
distributions; these diagnostics do not identify a unique physical cause.

**Regression checks.** Preserve the split-10 measured difference to 0.004 dex, and the near-invariance of
the 300-star coefficients under the scatter exchange to 0.003 dex. These are checks on this fixed dataset.

In [ ]:
check = size_results[(size_results.seed == 10) & (size_results.split == 'validation')].set_index('model')
assert abs(check.loc['full', 'delta'] - 0.0114) < 0.004
assert abs(check.loc['300 theta, full scatter', 'delta']) < 0.003
tab10 = size_tables[(size_tables.seed == 10) & (size_tables.split == 'validation')]
full = tab10[tab10.model == 'full'].set_index('label').loc[names[3:]]
small = tab10[tab10.model == '300'].set_index('label').loc[names[3:]]
assert (small.rmse > full.rmse).sum() >= 4, 'the narrower core does not uniformly improve RMSE'

## C.2 Lux, ridge and The Cannon with training-only continuum pixels

Use split 10 from C.1. Tune ridge on validation labels, then report all models on the same stars (Figure C.2).
The tables
include metallicity bins; sparse bins should be interpreted cautiously. Ridge's validation score has been used
to choose its penalty and is optimistic. We do not use its test score for this choice.

In [ ]:
idx, val, F, E, Ft, Et = [base[k] for k in ['idx', 'val', 'F', 'E', 'Ft', 'Et']]
refs = [L[val], truth]
eval_flux = [(F[val], E[val]), (Ft, Et)]


def fit_predict_lux(rows, labs, errors):
    A, B, s, Z, history = lux_train(labs, errors, F[rows], E[rows])
    out = [lux_test(f, e, A, B, s) for f, e in eval_flux]
    assert all(np.all(o[3] == 0) for o in out), 'labels depend on unconstrained spectral directions'
    assert all(np.all(np.isfinite(o[0])) for o in out)
    return [o[0] for o in out], (A, B, s)


def tuned_ridge(rows, labs):
    """Fit each available output on its labelled stars; choose one penalty on clean validation labels.
    Missing [N/Fe] excludes a star only from that output, not from the other eleven fits.
    """
    X = np.where(np.isfinite(E[rows]), F[rows], 1.0)
    Xeval = [np.where(np.isfinite(e), f, 1.0) for f, e in eval_flux]
    predictions = {}
    # Group outputs with the same missing-data pattern to reuse each matrix solve.
    patterns = {}
    for k in range(labs.shape[1]):
        good = np.isfinite(labs[:, k])
        patterns.setdefault(good.tobytes(), (good, []))[1].append(k)
    for lam in [1e-2, 1e-1, 1, 10]:
        pred = [np.zeros_like(ref) for ref in refs]
        for good, cols in patterns.values():
            x, y = X[good], labs[good][:, cols]
            mu, ym = x.mean(axis=0), y.mean(axis=0)
            x = x - mu
            coef = x.T @ np.linalg.solve(x @ x.T + lam * np.eye(len(x)), y - ym)
            for p, xe in zip(pred, Xeval):
                p[:, cols] = (xe - mu) @ coef + ym
        predictions[lam] = pred
    spread = np.diff(np.nanpercentile(labs, [2.5, 97.5], axis=0), axis=0)[0]
    lam = min(predictions, key=lambda l: np.median(robust_std(predictions[l][0] - refs[0]) / spread))
    return predictions[lam], lam


lux_pred, lux_model = fit_predict_lux(idx, L[idx], LE[idx])
ridge_pred, ridge_lam = tuned_ridge(idx, L[idx])
baseline = {'Cannon': base['pred'], 'Lux': lux_pred, 'ridge': ridge_pred}
print('Clean-label ridge penalty:', ridge_lam)
summary = []
for model, preds in baseline.items():
    for split, pred, ref in zip(['validation', 'test'], preds, refs):
        for lo, hi in [(-np.inf, np.inf), (-1.5, -0.5), (-0.5, 0), (0, 0.5)]:
            use = (ref[:, 2] >= lo) & (ref[:, 2] < hi)
            if use.sum() < 10:
                continue
            summary.append(statistics(pred[use], ref[use]).assign(model=model, split=split,
                            metallicity=f'[{lo}, {hi})', stars=use.sum()))
summary = pd.concat(summary, ignore_index=True)
show_table(summary, 'Baseline comparisons by label and metallicity bin (K, dex)')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k, label in zip(axes, [0, 1, None], ['T_eff [K]', 'log g [dex]', 'median [X/Fe] [dex]']):
    for j, (split, ref) in enumerate(zip(['validation', 'test'], refs)):
        scores = [robust_std(pred[j] - ref) for pred in baseline.values()]
        ax.bar(np.arange(3) + (j - 0.5) * 0.35,
               [s[k] if k is not None else np.median(s[3:]) for s in scores], 0.35, label=split)
    ax.set_xticks(range(3), list(baseline))
    ax.set(ylabel='robust scatter, ' + label)
axes[0].legend()
plt.tight_layout()
plt.show()

**Figure C.2.** Robust scatter about ASPCAP for temperature (kelvin), gravity (dex), and the median over nine
abundances (dex), using continuum pixels selected from training stars only. Blue bars show high-S/N validation
stars and orange bars the S/N 50–100 test sample. Compare this ranking with Figure 11.1; the accompanying tables
give bias and RMSE as well as scatter, separately for metallicity bins.

## C.3 Missing and noisy labels, repeated

Figure C.3 compares three draws in which nitrogen is hidden for half the training stars. Compare Lux using all
available data with
Lux using only complete stars, Cannon using complete stars, and ridge using every available label separately.
Then add noise of five times the catalogue error to all training labels. Retune ridge for that draw, and fit
Lux both with the correct total errors and with the original, underestimated errors. Each comparison uses
identical perturbations for all methods. Held-out labels and spectra are never perturbed. Continuum selection
is held fixed from C.1 to isolate label fitting; each model's label scaling uses only its available, perturbed
training labels. This does not simulate re-estimating the continuum from a noisy reference catalogue.

In [ ]:
experiments, experiment_tables = [], []
for seed in [20, 21, 22]:
    rng = np.random.default_rng(seed)
    missing = rng.random(len(idx)) < 0.5
    labs = L[idx].copy()
    labs[missing, names.index('n_fe')] = np.nan
    complete = idx[~missing]
    cm = fit_cannon(complete, L[complete], F, E)
    preds = {'Lux all': fit_predict_lux(idx, labs, LE[idx])[0],
             'Lux complete': fit_predict_lux(complete, L[complete], LE[complete])[0],
             'Cannon complete': [predict_cannon(cm, f, e)[0] for f, e in eval_flux]}
    preds['ridge available'], lam = tuned_ridge(idx, labs)
    print('Missing draw', seed, 'ridge penalty', lam, flush=True)
    noisy = L[idx] + rng.normal(size=L[idx].shape) * 5 * LE[idx]
    cm = fit_cannon(idx, noisy, F, E)
    noise_preds = {'Cannon': [predict_cannon(cm, f, e)[0] for f, e in eval_flux],
                   'Lux correct errors': fit_predict_lux(idx, noisy, np.sqrt(26) * LE[idx])[0],
                   'Lux underestimated errors': fit_predict_lux(idx, noisy, LE[idx])[0]}
    noise_preds['ridge retuned'], lam = tuned_ridge(idx, noisy)
    print('Noise draw', seed, 'ridge penalty', lam, 'elapsed seconds', round(time.time() - t0), flush=True)
    for experiment, collection in [('missing', preds), ('noisy', noise_preds)]:
        for model, predictions in collection.items():
            for j, (split, ref) in enumerate(zip(['validation', 'test'], refs)):
                reference = preds['Lux complete'][j] if experiment == 'missing' else baseline[
                    'Lux' if model.startswith('Lux') else ('ridge' if model.startswith('ridge') else 'Cannon')][j]
                delta, ci = compare(predictions[j], reference, ref)
                k = names.index('n_fe') if experiment == 'missing' else -1
                experiments.append(dict(experiment=experiment, seed=seed, model=model, split=split,
                                        delta=delta[k], lo=ci[0, k], hi=ci[1, k]))
                experiment_tables.append(statistics(predictions[j], ref).assign(
                    experiment=experiment, seed=seed, model=model, split=split))
experiments = pd.DataFrame(experiments)
experiment_tables = pd.concat(experiment_tables, ignore_index=True)
print(experiments.round(5).to_string(index=False))
show_table(experiment_tables, 'Per-label results for every perturbation draw (K, dex)')
assert np.isfinite(experiments[['delta', 'lo', 'hi']]).all().all()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for row, experiment in enumerate(['missing', 'noisy']):
    for col, split in enumerate(['validation', 'test']):
        ax = axes[row, col]
        d = experiments[(experiments.experiment == experiment) & (experiments.split == split)]
        y = np.arange(len(d))
        ax.hlines(y, d.lo, d.hi, color='C0')
        ax.plot(d.delta, y, 'o', color='C0')
        ax.set_yticks(y, [f'{r.model} (draw {r.seed})' for r in d.itertuples()])
        ax.axvline(0, color='k', lw=0.8)
        ax.set(title=experiment + ', ' + split, xlabel='change in robust scatter [dex]')
plt.tight_layout()
plt.show()

**Figure C.3.** Missing nitrogen (top) and added label noise (bottom), for validation stars (left) and low-S/N
test stars (right), with three independent perturbation draws. The horizontal axis is the change in robust
scatter in dex: nitrogen relative to Lux trained on complete stars in the top row; median abundance scatter
relative to each method's clean-label baseline in the bottom row. Points show estimates and bars paired 95%
bootstrap intervals. Negative values mean improvement. Compare correct and underestimated errors for Lux,
and the retuned ridge control, before attributing robustness specifically to the use of label uncertainties.

None of the three missing-label draws establishes a nitrogen improvement from the extra, incompletely labelled
stars. For added noise, The Cannon's median abundance scatter increases by roughly 0.05–0.07 dex; Lux changes
by less than 0.01 dex in these draws, even with underestimated errors. Retuning ridge matters and its change
varies with the draw. The comparison supports Lux's robustness here, but does not isolate accurate error bars
as its cause: treating labels as outputs and the latent representation also differ from The Cannon.

**Regression checks.** For this fixed sample and noise level, allow a broad 0.03 dex lower bound on Cannon's
degradation and a 0.02 dex upper bound on the absolute Lux change, on both held-out samples.

In [ ]:
noise = experiments[experiments.experiment == 'noisy']
assert (noise[noise.model == 'Cannon'].delta > 0.03).all()
assert (noise[noise.model.str.startswith('Lux')].delta.abs() < 0.02).all()

## C.4 Predicting spectra and measuring labels are different questions

The Cannon predicts a spectrum from labels. Lux with 49 latents and 12 labels does not define a unique such
prediction: labels constrain at most 12 latent directions. Below we choose the minimum-norm latent solution
from labels alone and show explicitly that adding a null-space direction preserves the labels but changes
the spectrum. Its spectral residual is conditional on that arbitrary convention; it is not a fair ranking
of the two models' ability to predict spectra. Fitting latents to the very spectrum being predicted would
instead measure reconstruction. A predictive distribution would need an additional, specified latent prior.

Label response is well defined when spectra constrain the needed directions (Figure C.4). For a representative
validation
star compare d[Mg/Fe]/dflux for both methods, using the same pixels, and verify each response against a small
numerical flux perturbation. The Cannon response uses the local Jacobian approximation; the finite difference
checks its adequacy at this star, rather than assuming it is exact for nonlinear residuals.

In [ ]:
A, B, s_lux = lux_model
th, s_can, pivot, scale = base['model']
ref = refs[0]
z_labels = ref @ np.linalg.pinv(A).T
predicted_lux = z_labels @ B.T
predicted_cannon = vectorize_n((ref - pivot) / scale) @ th.T
_, _, vh = np.linalg.svd(A, full_matrices=True)
null = vh[-1]
assert np.max(np.abs(A @ null)) < 1e-8
assert np.linalg.norm(B @ null) > 1e-6, 'chosen latent direction must change spectra'
good = np.isfinite(E[val]) & (th[:, 0] != 0)
variance = E[val]**2 + s_can**2
for name, prediction in [('Cannon', predicted_cannon), ('Lux, minimum-norm label latents', predicted_lux)]:
    q = np.sum(np.where(good, (F[val] - prediction)**2 / variance, 0), axis=1) / good.sum(axis=1)
    print(name, 'median spectral chi-squared per pixel:', np.median(q))
n = np.argmin(np.abs(ref[:, 0] - np.median(ref[:, 0])))
f, e = F[val][n], E[val][n]
ok = np.isfinite(e) & (th[:, 0] != 0)
mg = names.index('mg_fe')
fit, _ = predict_cannon(base['model'], f[None], e[None])
x = (fit[0] - pivot) / scale
J = th[ok] @ dvectorize_n(x)
w = 1 / (e[ok]**2 + s_can[ok]**2)
response_c = (np.linalg.pinv((J.T * w) @ J) @ (J.T * w))[mg] * scale[mg]
w_lux = 1 / (e[ok]**2 + s_lux[ok]**2)
response_l = (A @ np.linalg.pinv((B[ok].T * w_lux) @ B[ok], rcond=1e-12) @ (B[ok].T * w_lux))[mg]
# A symmetric perturbation avoids the first-order error of a forward finite difference.
p = np.argmax(np.abs(response_l))
pixel = np.flatnonzero(ok)[p]
h = 1e-5
fplus, fminus = f.copy(), f.copy()
fplus[pixel] += h
fminus[pixel] -= h
num_l = (lux_test(fplus[None], e[None], A, B, s_lux)[0][0, mg] -
         lux_test(fminus[None], e[None], A, B, s_lux)[0][0, mg]) / (2 * h)
num_c = (predict_cannon(base['model'], fplus[None], e[None], [x])[0][0, mg] -
         predict_cannon(base['model'], fminus[None], e[None], [x])[0][0, mg]) / (2 * h)
print('Response at pixel', pixel, '(analytic, finite difference): Lux', response_l[p], num_l,
      '; Cannon', response_c[p], num_c)
assert np.isclose(num_l, response_l[p], rtol=1e-3, atol=1e-4)
assert np.isclose(num_c, response_c[p], rtol=0.2, atol=0.01), 'local Jacobian approximation at this star'
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
window = (wave > 15730) & (wave < 15785)
axes[0].plot(wave[window], (B @ null)[window], label='flux change along one label-null direction')
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set(ylabel='change in normalised predicted flux')
axes[0].legend()
for response, label in [(response_c, 'Cannon'), (response_l, 'Lux')]:
    window_ok = window[ok]
    axes[1].plot(wave[ok][window_ok], response[window_ok], label=label)
axes[1].set(xlabel='vacuum wavelength [Å]', ylabel='d[Mg/Fe]/dflux [dex per unit flux]')
axes[1].legend()
plt.tight_layout()
plt.show()

**Figure C.4.** A validation giant near the median temperature, in a window around Mg I. Top: two different
Lux predictions differ by the plotted flux change when one unit null-space latent vector is added, although
their labels are identical; predicting spectra from labels therefore requires an extra convention. Bottom:
local response of inferred [Mg/Fe]
(dex per unit normalised flux) to each pixel, for Cannon and Lux. The horizontal axis is vacuum wavelength in
angstroms. These are inverse responses, not the forward flux derivatives plotted in earlier chapters.

## References

This appendix extends the models and experimental controls in Chapters 3, 10 and 11:
- Ness et al. 2015, [The Cannon](https://doi.org/10.1088/0004-637X/808/1/16).
- Horta et al. 2025, [Lux](https://arxiv.org/abs/2502.01745).


In [ ]:
print('Total runtime:', round(time.time() - t0), 'seconds')